# LFwC v2.0

## Preparations

Below you will find preparatory stuff such as imports and constant definitions for use down the road.

### Imports

In [ ]:
import json
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import rc
from matplotlib.colors import ListedColormap
from matplotlib.ticker import ScalarFormatter
from packaging.version import Version, parse
import colorcet as cc

### Constants

In [ ]:
CORPUS_PATH: Path = Path("../../public_data/lfwc-v2.0-masked.csv")
FIGURE_DEST: Path = Path("../../figures")

Y_LABELS: list[str] = [
    "Ubiquiti",
    "TRENDnet",
    "NETGEAR",
    "Linksys",
    "EnGenius",
    "EDIMAX",
    "D-Link",
    "ASUS",
    "TP-Link",
    "FRITZ!",
    "DrayTek",
    "MikroTik",
    "Moxa",
    "Netis",
    "TOTOLINK",
    "Zyxel"
]
Y_LABELS.sort()
GLASBEY = ListedColormap(cc.glasbey[:20])

### Matplotlib Settings

In [ ]:
rc("font", **{"family": "serif", "serif": ["Times"], "size": 15})
rc("text", usetex=True)
pd.set_option("display.max_colwidth", None)

### Read Data

In [ ]:
df = pd.read_csv(CORPUS_PATH, index_col=0)

## Peek Into Raw Data

In [ ]:
df

## Corpus Statistics Overview

In [ ]:
def corpus_statistics_overview(df: pd.DataFrame) -> pd.DataFrame:
    df_stats: pd.DataFrame = (
        df.groupby(["manufacturer"], as_index=False)
        .nunique()[["manufacturer", "sha256", "device_name"]]
        .rename(columns={"manufacturer": "Manufact.", "sha256": "Samples", "device_name": "Devices"})
    )

    df_stats["Mean Samples per Device"] = (df_stats["Samples"] / df_stats["Devices"]).round(2)
    df_stats["Mean Size per Sample"] = (
        df[["manufacturer", "compressed_firmware_size"]]
        .groupby(["manufacturer"], as_index=False)
        .mean()["compressed_firmware_size"]
        / 1024**2
    ).round(0)

    df_stats["Mean Files per Sample"] = (
        df[["manufacturer", "files_in_firmware"]].groupby(["manufacturer"], as_index=False).mean()["files_in_firmware"]
    ).round(2)

    return df_stats

In [ ]:
df_stats: pd.DataFrame = corpus_statistics_overview(df)
df_stats

In [ ]:
df_stats["Samples"].sum()

## Firmware distribution per release date

In [ ]:
def firmware_distribution_per_release_date(df: pd.DataFrame) -> None:
    df_removed_day_from_date = df.copy()
    rc("font", **{"family": "serif", "serif": ["Times"], "size": 16})
    df_removed_day_from_date["release_date"] = (
        df_removed_day_from_date["release_date"].str.split("-").str[:-2].str.join("-")
    )
    df_history = (
        df_removed_day_from_date.groupby(["release_date", "manufacturer"], as_index=False)
        .nunique()
        .pivot(index="release_date", columns=["manufacturer"], values="md5")
        .fillna(value=0.0)
    )

    ax = df_history.plot(
        kind="bar",
        grid=True,
        stacked=True,
        logy=False,
        figsize=(8, 6),
        rot=50,
        legend=False,
        edgecolor="black",
        cmap=GLASBEY,
    )
    ax.set_xticklabels(["unk."] + [str(i) for i in range(2004, 2026)])
    ax.set_ylabel("Samples")
    ax.set_xlabel("Release Year")
    ax.set_ylim(0, 1500)
    ax.legend(ncols=2, fontsize=13)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(ScalarFormatter())
    plt.tight_layout()
    rc("font", **{"family": "serif", "serif": ["Times"], "size": 15})
    plt.show()

In [ ]:
firmware_distribution_per_release_date(df)

## Distribution of device classes

In [ ]:
def distribution_of_device_classes(df: pd.DataFrame) -> None:
    df_corpus_misc_classes = df.copy()

    flt = df_corpus_misc_classes["device_class"].str.contains(
        "Accesspoint|controller|board|converter|encoder|gateway|kvm|media|nas|phone|power_supply|printer|recorder|san|wifi-usb"
    )
    df_corpus_misc_classes.loc[flt, "device_class"] = "misc"

    by_classes = (
        df_corpus_misc_classes.groupby(["device_class", "manufacturer"], as_index=False)
        .nunique()
        .pivot(index="device_class", columns=["manufacturer"], values="md5")
        .fillna(value=0.0)
    )

    rc("font", **{"family": "serif", "serif": ["Times"], "size": 18})
    ax = by_classes.plot(
        kind="bar",
        grid=True,
        stacked=False,
        logy=True,
        figsize=(20, 4.5),
        cmap=GLASBEY,
        edgecolor="black",
        legend=False,
        width=0.8,
        rot=0,
    )
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.set_ylabel("Samples")
    ax.set_xlabel("Device Class")
    ax.set_xlim(-0.41, 8.49)
    ax.legend(ncols=9, fontsize=16, bbox_to_anchor=(1.05, 1.3))
    for i in range(0, 11):
        ax.axvline(i + 0.500, color="black", linewidth=1)
    rc("font", **{"family": "serif", "serif": ["Times"], "size": 15})
    plt.show()

In [ ]:
distribution_of_device_classes(df)

## Detected Linux kernel banners

In [ ]:
def kernel_banners(df: pd.DataFrame) -> None:
    df_linux_prep = df.copy()
    linux_series = df_linux_prep[df_linux_prep["linux_banners"].notnull()]["linux_banners"].apply(
        lambda x: x.split("|")
    )
    df_linux_prep["linux_banners"] = linux_series
    df_linux_prep = df_linux_prep.explode("linux_banners", ignore_index=True)

    def bucketize(ver_str):
        if isinstance(ver_str, float):
            return parse("0.0")
        prepared = ver_str.split(" ")[-1].split(".")[0:2]
        # filter false positives
        if int(prepared[0]) == 1:
            return parse("0.0")
        if int(prepared[0]) <= 2 and int(prepared[1]) <= 2:
            return parse("0.0")

        ver = parse(".".join(prepared))
        return ver

    df_bucketize_version = df_linux_prep.copy()
    df_bucketize_version["linux_banners"] = df_bucketize_version["linux_banners"].apply(bucketize)
    df_bucketize_version
    rc("font", **{"family": "serif", "serif": ["Times"], "size": 22})
    df_linux_banners = (
        df_bucketize_version.groupby(["linux_banners", "manufacturer"], as_index=False)
        .nunique()
        .pivot(index="linux_banners", columns=["manufacturer"], values="md5")
        .fillna(value=0.0)
    )

    ax = df_linux_banners.plot(
        kind="barh",
        grid=True,
        stacked=True,
        logx=True,
        figsize=(21, 9),
        rot=0,
        legend=False,
        edgecolor="black",
        color=["grey"],
    )
    ax.set_ylabel(None)
    ax.set_xlabel("Linux Kernel Version Banners [Major.Minor, log]")
    ax.set_yticklabels(["unk."] + ax.get_yticklabels()[1:], ha="left", va="center", position=(-0.0275, 0))
    ax.set_axisbelow(True)
    for i in range(0, 23):
        x = df_linux_banners.iloc[i].sum()
        plt.text(x + 10, i, int(x), va="center")
    ax.xaxis.set_major_formatter(ScalarFormatter())
    plt.tight_layout()
    plt.show()

In [ ]:
kernel_banners(df)

## ISAs

In [ ]:
def isa_distribution(df: pd.DataFrame) -> None:
    df_arch_prep = df.copy()
    arch_series = df_arch_prep[df_arch_prep["elf_architectures"].notnull()]["elf_architectures"].apply(
        lambda x: x.split("|")
    )
    df_arch_prep["elf_architectures"] = arch_series
    df_arch_prep

    rc("font", **{"family": "serif", "serif": ["Times"], "size": 18})

    by_arch = df_arch_prep.explode("elf_architectures", ignore_index=True)
    by_arch = (
        by_arch.groupby(["elf_architectures", "manufacturer"], as_index=False)
        .nunique()
        .pivot(index="elf_architectures", columns=["manufacturer"], values="md5")
        .fillna(value=0.0)
    )
    ax = by_arch.plot(
        kind="bar",
        grid=True,
        stacked=False,
        logy=True,
        figsize=(20, 4.5),
        cmap=GLASBEY,
        edgecolor="black",
        legend=False,
        width=0.8,
        rot=0,
    )
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.set_ylabel("Samples [log]")
    ax.set_xlabel("Instruction Set Architecture")
    ax.set_xlim(-0.5, 8.5)
    for i in range(0, 11):
        ax.axvline(i + 0.505, color="black", linewidth=1)
    ax.legend(ncols=9, fontsize=16, bbox_to_anchor=(1.05, 1.3))
    rc("font", **{"family": "serif", "serif": ["Times"], "size": 15})
    plt.show()

In [ ]:
isa_distribution(df)

In [ ]:
df_arch_prep = df.copy()
arch_series = df_arch_prep[df_arch_prep["elf_architectures"].notnull()]["elf_architectures"].apply(
    lambda x: x.split("|")
)
df_arch_prep["elf_architectures"] = arch_series
df_arch_prep

In [ ]:
test = set({})
for row in arch_series:
    for arch in row:
        test.add(arch)

In [ ]:
df_arch_prep[(df_arch_prep["release_date"] > "2023-07-01") & (df_arch_prep["corpus_version"] == "v1.0")]

In [ ]:
df_arch_prep["device_class"].unique()

In [ ]:
for x in df_arch_prep["manufacturer"].unique():
    print(x, end=", ")